# Vector Stores

A vector store stores embeddings alongside their original text and metadata. It builds an index which allows fast ANN (approximate nearest neighbour) search: given a query vector, find the k closes document chunk vectors. This gives retrieval a time complexity of O(log n), whereas scanning the entire collection of vectors would be O(n).

## Local vs cloud-hosted

For this project we'll build a vector store using Chroma, as this runs locally. This can store up to 1 million vectors, whereas hosted options such Vertex AI Vector Search (Google), Pinecone and Weaviate can store hundreds of millions - billions.

# Features of Chroma

- supports filtering retrieved results by metadata (e.g page number)
- easily integrates with LangChain
- stores the raw text documents and their embeddings together


## Integrating Chroma with LangChain

We can embed the document chunks and write them to a local Chroma vector store in a single step using `Chroma.from_documents()` from `langchain_chroma`, by passing in our document chunks and the embedding model.

`from_documents()` calls `embedding_model.embed_documents([chunk.page_content for chunk in chunks])` internally, pairs each vector with its chunk text and metadata and writes everything to the local vector store.

In [ ]:
from langchain_chroma import Chroma
from rag_pipeline.vector_store import create_embedding_model

embedding_model = create_embedding_model()

In [ ]:
vectorstore = Chroma.from_documents(
    documents=chunks,  # placeholder for this example
    embedding=embedding_model,
    persist_directory="chroma_db",
)

In [ ]:
# load the existing vector store (doesn't re-build it)

vectorstore = Chroma(
    persist_directory="chroma_db",
    embedding_function=embedding_model,
)

# Inspecting the vector store

After running `run_indexing.py`, we should inspect the vector store that was created.

In [ ]:
from rag_pipeline.vector_store import load_vector_store, create_embedding_model

store = load_vector_store(create_embedding_model())  # This returns a Chroma object
print(store._collection.count())

This outputs `262` which matches the number of chunks that were printed in the logs.

In [ ]:
sample = store.get(limit=3, include=["documents", "metadatas"])
for doc, metadata in zip(sample["documents"], sample["metadatas"]):
    print(metadata)
    print(doc[:100])
    print()

As expected, we see each chunk has metadata stored with it, which consists of `chunk_index`, `source`, `page` and `total_pages`.

# Test similarity search

In [ ]:
from rag_pipeline.retrieval import create_retriever

retriever = create_retriever(store, embedding_model, k=5)
docs = retriever.invoke("What is the maximum payout for emergency medical expenses?")
docs

The retrieved chunks appear to be relevant to the query, which gives us some initial confidence that the indexing pipeline is working as expected.

# LangChain vector store functionality

Note that in the above sections we use `load_vector_store()` to return a Chroma object. Even if we chose to use a different vector store provider (e.g Pinecone or pgvector), it would still be integrated with LangChain in the same way, i.e we could call the vector store object in the same way.

In the above sections we have called the following methods of our Chroma vector store object:

- store._collection.count(): gets the number of document chunks
- store.get(limit include)
- store.similarity_search(query, k)

LangChain vector stores also have the following methods:

- store.similarity_search_with_score(query, k): lower sccore = smaller cosine distance = more similar
- store.max_marginal_relevance_search(query, k, fetch_k): (MMR) re-ranks the k candidates to maximise diversity and reduce redundant results
- store.add_documents(new_chunks): add new document chunks to an existing store

`store.as_retriever(search_type="similarity", search_kwargs={"k": k})` returns a `VectoreStoreRetriever`. This will be used in the retrieval pipeline. The following search types are available:

#### `similarity` (default)

- Chroma uses a graph-based ANN algorithm called Hierarchical Navigable Small World (HNSW)
- This ranks all vectors by cosine similarity to the query and returns the top k. 
- This has the benefit of being simple, fast and deterministic, however if there is a lot of repetition in the documents then the retrieved chunks could be near-duplicates.

#### `mmr` (maximum marginal relevance)

- This fetches a larger pool of `fetch_k` chunks by similarity, then iteratively selects chunks that are both relevant to the query and different from what's already been selected. 
- `lambda_mult` controls how different the retrieved chunks should be from one another. 
- This is slower than pure similarity search.
- Lambda_mult needs to be tuned, and low values can select chunks that are technically diverse but less relevant

#### `similarity_score_threshold`

- Returns all chunks whose similarity score exceeds a threshold, up to k
- Acts as a quality gate: if nothing is relevant enough, nothing is returned
- If threshold is too high, you risk returning nothing on valid documents
- Set threshold by testing what scores relevant vs irrelevant chunks return
- Chroma internally uses cosine distance, which LangChain normalises into a 0-1 relevance score, so score_threshold=0.5 means 'at least 50% similar'

For this project we'll start with the default `similarity` search type due to it being simple, fast and deterministic.

We could consider `mmr` in the future if we observe that irrelevant chunks re being retrieved. This method would require extra work to tune `lambda_mult`. `similarity_score_threshold` also has the benefit of improving chunk relevance.